# P7 Kaggle Training Notebook
This notebook runs the P7 (MHSDF) baseline on the fully preprocessed MMHS150K dataset in Kaggle.

This version uses only:
- Stage 1: binary classification (hate vs not-hate)
- Stage 2: 5-class hate-type classification (Racist, Sexist, Homophobe, Religion, OtherHate)

It does not run the 6-class setup.

Before running, upload this repository to Kaggle or clone it into `/kaggle/working/` so the `scripts/` and `src/` packages are available.

## 1. Import Kaggle-Compatible Libraries
Core libraries for data handling, modeling, visualization, and filesystem access.

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)

## 2. Load Preprocessed Dataset from Kaggle Input
Set the dataset path from `/kaggle/input` and link it into the repo so existing scripts use it.

In [ ]:
from pathlib import Path
import sys

# Candidate repo locations on Kaggle
candidates = [
    Path("/kaggle/working/Memes_Vibe_Classifier"),
    Path.cwd(),
]
# include any top-level dataset folders under /kaggle/input
input_root = Path("/kaggle/input")
if input_root.exists():
    for p in input_root.iterdir():
        candidates.append(p / "Memes_Vibe_Classifier")
# pick first existing candidate that has a src/ package
REPO_DIR = next((p for p in candidates if p.exists() and (p / "src").exists()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Repo not found in expected locations; clone or upload it to /kaggle/working or add as notebook input.")
DATASET_INPUT_DIR = Path("/kaggle/input/updated-hate-speech-dataset/dataset")  # adjust if your dataset name differs

sys.path.insert(0, str(REPO_DIR))
repo_dataset = REPO_DIR / "dataset"
if repo_dataset.exists() and not repo_dataset.is_symlink():
    print(f"Repo dataset exists: {repo_dataset}")
else:
    if repo_dataset.exists():
        repo_dataset.unlink()
    repo_dataset.symlink_to(DATASET_INPUT_DIR, target_is_directory=True)
print("REPO_DIR:", REPO_DIR)
print("Dataset:", repo_dataset.resolve())

## 3. Verify Schema and Basic Stats
Check counts and basic structure for GT, splits, images, and OCR.

In [ ]:
from collections import Counter

GT_PATH = repo_dataset / 'MMHS150K_GT.json'
SPLITS_DIR = repo_dataset / 'splits'
IMG_DIR = repo_dataset / 'img_resized'
OCR_PATH = repo_dataset / 'ocr_consolidated.json'
LABELS_PATH = repo_dataset / 'processed_labels.json'

gt = json.load(open(GT_PATH, encoding='utf-8'))
print('GT entries:', len(gt))
sample_key = next(iter(gt))
print('Sample entry keys:', list(gt[sample_key].keys()))

splits = {}
for split in ['train', 'val', 'test']:
    split_path = SPLITS_DIR / f'{split}_ids.txt'
    splits[split] = [x.strip() for x in open(split_path, encoding='utf-8') if x.strip()]
    print(f'{split} IDs:', len(splits[split]))

img_count = len(list(IMG_DIR.glob('*.jpg')))
print('Images:', img_count)

if OCR_PATH.exists():
    ocr = json.load(open(OCR_PATH, encoding='utf-8'))
    print('OCR entries:', len(ocr), '| with text:', sum(1 for v in ocr.values() if v and str(v).strip()))
else:
    print('OCR consolidated not found')

if LABELS_PATH.exists():
    labels = json.load(open(LABELS_PATH, encoding='utf-8'))
    print('Processed labels:', len(labels))
    dist = Counter(v['hard_label_6class'] for v in labels.values())
    print('Label dist:', dict(sorted(dist.items())))
else:
    print('processed_labels.json not found')

## 4. Train/Validation/Test Split
Use the official split files and persist them for reproducibility.

In [ ]:
splits_cache = {k: v for k, v in splits.items()}
cache_path = Path('/kaggle/working/splits_cache.json')
with open(cache_path, 'w', encoding='utf-8') as f:
    json.dump(splits_cache, f)
print('Saved splits cache ->', cache_path)

## 5. Define Baseline Model
Reference the P7 MHSDF model (ResNet18 + BiLSTM). This notebook uses stage 1 (binary) and stage 2 (5-class hate-only).

In [ ]:
from src.models.mhsdf import MHSDF

def build_p7_model(vocab_size: int, num_classes: int):
    return MHSDF(vocab_size=vocab_size, num_classes=num_classes, multilabel=False, freeze_cnn=True)

print('P7 model class ready. Training script will build the vocab and instantiate this model.')

## 6. Train Model with Checkpointing and Logging
Run P7 training. Default is stage 1 (binary). Checkpoints and CSV logs are saved under `checkpoints/<run_name>/`.

In [ ]:
import subprocess

# Training config
RUN_NAME = 'p7_full_stage1'
EPOCHS = 10
BATCH_SIZE = 32
NUM_WORKERS = 4
STAGE = 1  # 1=binary (hate vs not-hate), 2=hate-only 5-class
MAX_LEN = 64
LR = 1e-3
SAVE_EVERY = 0  # set >0 for step checkpoints

cmd = [
    'python', 'scripts/train_p7.py',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--stage', str(STAGE),
    '--max-len', str(MAX_LEN),
    '--lr', str(LR),
    '--run-name', RUN_NAME,
    '--save-every', str(SAVE_EVERY),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 6b. Run Stage 2 Automatically
Stage 2 (5-class hate-only) will run after Stage 1.

In [ ]:
# Run stage 2 (hate-only 5-class) after stage 1
RUN_NAME_STAGE2 = 'p7_full_stage2'
cmd_stage2 = [
    'python','scripts/train_p7.py','--epochs','5','--batch-size','32','--num-workers','4','--stage','2',
    '--max-len','64','--lr','1e-3','--run-name', RUN_NAME_STAGE2,'--save-every','0'
]
print('Running:', ' '.join(cmd_stage2))
subprocess.run(cmd_stage2, check=True)

## 7. Persist Intermediate Results
Copy logs and checkpoints into `/kaggle/working` for easy download.

In [ ]:
artifacts_dir = Path('/kaggle/working/p7_artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)

run_dirs = [
    REPO_DIR / 'checkpoints' / RUN_NAME,
    REPO_DIR / 'checkpoints' / RUN_NAME_STAGE2,
 ]
for run_dir in run_dirs:
    if run_dir.exists():
        dst = artifacts_dir / run_dir.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(run_dir, dst)
        print('Copied artifacts ->', dst)
    else:
        print('Run directory not found:', run_dir)

## 8. Evaluate on Test Set
Evaluate the best checkpoint on the test split and save metrics to `/kaggle/working`.

In [ ]:
eval_outputs = []
for run_name, stage in [(RUN_NAME, 1), (RUN_NAME_STAGE2, 2)]:
    best_ckpt = REPO_DIR / 'checkpoints' / run_name / 'best.pt'
    eval_out = Path('/kaggle/working') / f'p7_eval_test_stage{stage}.json'
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Best checkpoint not found: {best_ckpt}')
    cmd = [
        'python', 'scripts/eval_p7.py',
        '--checkpoint', str(best_ckpt),
        '--split', 'test',
        '--num-workers', str(NUM_WORKERS),
        '--output', str(eval_out),
]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    eval_outputs.append((stage, eval_out))

## 9. Compute and Log Metrics
Load evaluation metrics and save a compact summary for reporting.

In [ ]:
summaries = []
for stage, eval_out in eval_outputs:
    metrics = json.load(open(eval_out, encoding='utf-8'))
    summaries.append({
        'run_name': RUN_NAME if stage == 1 else RUN_NAME_STAGE2,
        'stage': stage,
        'metrics': metrics,
    })

summary_path = Path('/kaggle/working') / 'p7_metrics_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summaries, f, indent=2)
print('Saved metrics summary ->', summary_path)
for s in summaries:
    print('Stage', s['stage'], 'metrics:', s['metrics'])

## 10. Visualize Training Curves
Plot loss and selected metrics from the CSV log.

In [ ]:
for run_name, stage in [(RUN_NAME, 1), (RUN_NAME_STAGE2, 2)]:
    log_path = REPO_DIR / 'checkpoints' / run_name / 'training_log.csv'
    if not log_path.exists():
        print('Log not found:', log_path)
        continue
    df = pd.read_csv(log_path)
    print(run_name, df.head())

    plt.figure(figsize=(8, 4))
    plt.plot(df['step'], df['train_loss'], label='train_loss')
    plt.xlabel('step')
    plt.ylabel('loss')
    plt.title(f'Training Loss ({run_name})')
    plt.legend()
    loss_plot = Path('/kaggle/working') / f'{run_name}_loss.png'
    plt.tight_layout()
    plt.savefig(loss_plot)
    plt.show()
    print('Saved:', loss_plot)

    metric_key = 'binary/macro_f1' if stage == 1 else 'multiclass/macro_f1'
    metric_df = df[df['metric_key'] == metric_key]
    if not metric_df.empty:
        plt.figure(figsize=(8, 4))
        plt.plot(metric_df['epoch'], metric_df['metric_value'], label=metric_key)
        plt.xlabel('epoch')
        plt.ylabel('score')
        plt.title(f'Validation Metric ({run_name})')
        plt.legend()
        metric_plot = Path('/kaggle/working') / f'{run_name}_metric.png'
        plt.tight_layout()
        plt.savefig(metric_plot)
        plt.show()
        print('Saved:', metric_plot)
    else:
        print('Metric not found in log:', metric_key)

## 11. Visualize Key Predictions
Build a small confusion matrix on a subset of the test split.

In [ ]:
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Subset
from src.data.dataset import MMHS150KDataset
from src.utils.text_vectorizer import TextVectorizer
from src.models.mhsdf import MHSDF
from src.utils.config import DataConfig

MAX_PRED_SAMPLES = 2000

def target_for_stage(batch_item: dict, stage: int) -> int:
    if stage == 1:
        return int(batch_item['label_binary'])
    if stage == 2:
        return max(0, int(batch_item['label']) - 1)
    return int(batch_item['label'])

class StageCollator:
    def __init__(self, vectorizer: TextVectorizer, max_len: int, stage: int):
        self.vectorizer = vectorizer
        self.max_len = max_len
        self.stage = stage
    def __call__(self, batch):
        images = torch.stack([b['image'] for b in batch])
        texts = [(b.get('ocr_text', '') or '') + ' ' + (b.get('tweet_text', '') or '') for b in batch]
        ids_batch, lengths = self.vectorizer.encode(texts, max_len=MAX_LEN)
        text_ids = torch.tensor(ids_batch, dtype=torch.long)
        lengths_t = torch.tensor(lengths, dtype=torch.long)
        labels = torch.tensor([target_for_stage(b, stage=self.stage) for b in batch], dtype=torch.long)
        return {'image': images, 'text_ids': text_ids, 'lengths': lengths_t, 'label': labels}

cfg = DataConfig()
ds_raw = MMHS150KDataset(split='test', config=cfg)
indices = list(range(min(MAX_PRED_SAMPLES, len(ds_raw))))
ds = Subset(ds_raw, indices)

for run_name, stage in [(RUN_NAME, 1), (RUN_NAME_STAGE2, 2)]:
    best_ckpt = REPO_DIR / 'checkpoints' / run_name / 'best.pt'
    if not best_ckpt.exists():
        print('Missing checkpoint:', best_ckpt)
        continue
    ckpt = torch.load(best_ckpt, map_location='cpu')
    vocab = ckpt['vocab']
    num_classes = int(ckpt.get('num_classes', 6))
    vec = TextVectorizer(min_freq=1)
    vec.vocab = vocab
    collate = StageCollator(vectorizer=vec, max_len=MAX_LEN, stage=stage)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MHSDF(vocab_size=len(vocab), num_classes=num_classes, multilabel=False, freeze_cnn=True)
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    model.to(device)
    model.eval()

    y_true = []
    y_pred = []
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            text_ids = batch['text_ids'].to(device)
            lengths = batch['lengths'].to(device)
            labels = batch['label'].cpu().numpy().tolist()
            logits = model(images, text_ids, lengths)
            preds = logits.argmax(dim=-1).cpu().numpy().tolist()
            y_true.extend(labels)
            y_pred.extend(preds)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap='Blues')
    plt.title(f'Confusion Matrix (subset) - Stage {stage}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.colorbar()
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha='center', va='center', color='black')
    cm_path = Path('/kaggle/working') / f'{run_name}_confusion_matrix.png'
    plt.tight_layout()
    plt.savefig(cm_path)
    plt.show()
    print('Saved:', cm_path)

## 12. Save Final Artifacts
Bundle key outputs for easy download from Kaggle.

In [ ]:
final_dir = Path('/kaggle/working/p7_final')
final_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    Path('/kaggle/working') / 'p7_metrics_summary.json',
    Path('/kaggle/working') / f'{RUN_NAME}_loss.png',
    Path('/kaggle/working') / f'{RUN_NAME}_metric.png',
    Path('/kaggle/working') / f'{RUN_NAME}_confusion_matrix.png',
    Path('/kaggle/working') / f'{RUN_NAME_STAGE2}_loss.png',
    Path('/kaggle/working') / f'{RUN_NAME_STAGE2}_metric.png',
    Path('/kaggle/working') / f'{RUN_NAME_STAGE2}_confusion_matrix.png',
]
for p in files_to_copy:
    if p.exists():
        shutil.copy2(p, final_dir / p.name)
        print('Saved:', final_dir / p.name)
    else:
        print('Missing:', p)

print('Final artifacts folder:', final_dir)